In [ ]:
import duckdb
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

db_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq_warehouse.duckdb"
)

print("Database exists:", db_path.exists())

con = duckdb.connect(str(db_path), read_only=True)

print("Connected to warehouse successfully!")

In [ ]:
print(con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = 'warehouse'
      AND table_name IN (
          'dim_department',
          'dim_employee',
          'fact_headcount_monthly'
      )
    ORDER BY table_name
""").df())

In [ ]:
for name in [
    "headcount_monthly",
    "tier_counts",
    "workmode_counts"
]:
    print(name, "exists:", name in globals())

In [ ]:
from pathlib import Path

project_path = Path(r"C:\Users\Hp\Documents\pulseiq")
warehouse_db_path = project_path / "05_Starter" / "pulseiq_warehouse.duckdb"

print("Project folder exists:", project_path.exists())
print("Warehouse database exists:", warehouse_db_path.exists())
print("Notebook task:", "PQ-05 Workforce & Headcount Dashboard")

In [ ]:
import duckdb

con = duckdb.connect(
    r"C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq_warehouse.duckdb",
    read_only=True
)

tables = con.execute("SHOW TABLES").fetchdf()

print("Tables available in the warehouse:")
display(tables)

In [ ]:
from pathlib import Path

db_path = Path(
    r"C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq_warehouse.duckdb"
)

print("Database file exists:", db_path.exists())
print("Database file size (bytes):", db_path.stat().st_size if db_path.exists() else "Not found")

print("\nTables in the warehouse:")
print(con.execute("SHOW TABLES").fetchall())

print("\nTables in the source database:")
source_path = r"C:\Users\Hp\Documents\pulseiq\05_Starter\pulseiq.duckdb"

with duckdb.connect(source_path, read_only=True) as source_con:
    print(source_con.execute("SHOW TABLES").fetchall())

In [ ]:
print("Connected database:")
print(con.execute("SELECT current_database()").fetchall())

print("\nAttached databases:")
display(con.execute("PRAGMA database_list").fetchdf())

print("\nAvailable schemas:")
display(con.execute("""
    SELECT catalog_name, schema_name
    FROM information_schema.schemata
    ORDER BY catalog_name, schema_name
""").fetchdf())

print("\nAll visible tables and views:")
display(con.execute("""
    SELECT table_catalog, table_schema, table_name, table_type
    FROM information_schema.tables
    ORDER BY table_catalog, table_schema, table_name
""").fetchdf())

In [ ]:
required_tables = [
    "dim_employee",
    "fact_headcount_monthly",
    "dim_location"
]

for table_name in required_tables:
    result = con.execute(
        """
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = 'warehouse'
          AND table_name = ?
        """,
        [table_name]
    ).fetchdf()

    print(f"\n{table_name}:")
    display(result)

In [ ]:
tables_to_inspect = [
    "dim_employee",
    "fact_headcount_monthly",
    "dim_location"
]

for table_name in tables_to_inspect:
    print(f"\n{'=' * 60}")
    print(f"TABLE: warehouse.{table_name}")
    print("=" * 60)

    # Show column names and data types
    columns = con.execute(
        f"DESCRIBE warehouse.{table_name}"
    ).fetchdf()
    display(columns)

    # Show the first five records
    print(f"Sample records from {table_name}:")
    sample = con.execute(
        f"SELECT * FROM warehouse.{table_name} LIMIT 5"
    ).fetchdf()
    display(sample)

In [ ]:
import pandas as pd

# Load the three tables from the warehouse schema
employees = con.execute(
    "SELECT * FROM warehouse.dim_employee"
).fetchdf()

headcount_monthly = con.execute(
    "SELECT * FROM warehouse.fact_headcount_monthly"
).fetchdf()

locations = con.execute(
    "SELECT * FROM warehouse.dim_location"
).fetchdf()

print("EMPLOYEE DATA")
print("Rows:", len(employees))
print("Unique employee IDs:", employees["employee_id"].nunique())
print("\nCurrent-record flags:")
display(employees["is_current"].value_counts(dropna=False).to_frame("record_count"))

print("\nEmployment status:")
display(employees["employment_status"].value_counts(dropna=False).to_frame("record_count"))

print("\nHEADCOUNT DATA")
print("Rows:", len(headcount_monthly))
print("Date range:",
      headcount_monthly["month_start_date"].min(),
      "to",
      headcount_monthly["month_start_date"].max())

print("\nLOCATION DATA")
print("Rows:", len(locations))
display(locations)

print("\nWORK MODES")
display(employees["work_mode"].value_counts(dropna=False).to_frame("record_count"))

print("\nDEPARTMENT COUNTS IN MONTHLY DATA")
display(
    headcount_monthly.groupby("department_id")
    .size()
    .reset_index(name="monthly_records")
)

In [ ]:
# 1. Inspect the department lookup table
print("Department lookup columns:")
display(
    con.execute("DESCRIBE warehouse.dim_department").fetchdf()
)

print("\nDepartment lookup sample:")
display(
    con.execute("SELECT * FROM warehouse.dim_department LIMIT 15").fetchdf()
)

# 2. Find the latest month in the headcount fact table
latest_month = headcount_monthly["month_start_date"].max()

latest_data = headcount_monthly[
    headcount_monthly["month_start_date"] == latest_month
].copy()

print("\nLatest reporting month:", latest_month)

print("\nLatest month totals:")
print("Headcount at start:", latest_data["headcount_start"].sum())
print("Hires:", latest_data["hires"].sum())
print("Exits:", latest_data["exits"].sum())
print("Headcount at end:", latest_data["headcount_end"].sum())

print("\nMonthly department-level data for the latest month:")
display(
    latest_data[
        [
            "department_id",
            "headcount_start",
            "hires",
            "exits",
            "headcount_end",
            "attrition_rate_monthly_pct"
        ]
    ].sort_values("headcount_end", ascending=False)
)

In [ ]:
# Get the first and latest reporting months
first_month = headcount_monthly["month_start_date"].min()
latest_month = headcount_monthly["month_start_date"].max()

first_data = headcount_monthly[
    headcount_monthly["month_start_date"] == first_month
]

latest_data = headcount_monthly[
    headcount_monthly["month_start_date"] == latest_month
]

first_headcount = first_data["headcount_end"].sum()
latest_headcount = latest_data["headcount_end"].sum()

growth = latest_headcount - first_headcount

growth_pct = (
    (growth / first_headcount) * 100
    if first_headcount != 0 else None
)

print("HEADCOUNT GROWTH")
print("First reporting month:", first_month.strftime("%B %Y"))
print("First month-end headcount:", first_headcount)
print("Latest reporting month:", latest_month.strftime("%B %Y"))
print("Latest month-end headcount:", latest_headcount)
print("Net headcount change:", growth)
print(
    "Headcount growth (%):",
    round(growth_pct, 2) if growth_pct is not None else "Not calculable"
)

# Department mix for the latest reporting month
department_mix = latest_data[
    ["department_id", "headcount_end"]
].merge(
    con.execute("""
        SELECT department_id, department_name
        FROM warehouse.dim_department
    """).fetchdf(),
    on="department_id",
    how="left"
)

department_mix["share_pct"] = (
    department_mix["headcount_end"]
    / latest_headcount * 100
)

department_mix = department_mix.sort_values(
    "headcount_end", ascending=False
)

print("\nLATEST DEPARTMENT MIX")
display(department_mix)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

monthly_trend = (
    headcount_monthly
    .groupby("month_start_date", as_index=False)
    .agg(
        headcount=("headcount_end", "sum"),
        hires=("hires", "sum"),
        exits=("exits", "sum")
    )
    .sort_values("month_start_date")
)

start_count = monthly_trend.iloc[0]["headcount"]
end_count = monthly_trend.iloc[-1]["headcount"]
growth_pct = (end_count - start_count) / start_count * 100

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    monthly_trend["month_start_date"],
    monthly_trend["headcount"],
    marker="o",
    markersize=3,
    linewidth=2
)

ax.set_title(
    f"Headcount grew {growth_pct:.1f}%: "
    f"{int(start_count)} to {int(end_count)} employees",
    fontweight="bold"
)
ax.set_xlabel("Month")
ax.set_ylabel("Month-end headcount")
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(
    f"Insight: Headcount increased by {int(end_count - start_count)} "
    f"employees ({growth_pct:.2f}%) between "
    f"{monthly_trend.iloc[0]['month_start_date']:%B %Y} and "
    f"{monthly_trend.iloc[-1]['month_start_date']:%B %Y}."
)

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

print("Matplotlib version:", matplotlib.__version__)
print("Installation successful!")

## Headcount trend chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Summarize headcount across all departments for each month
monthly_trend = (
    headcount_monthly
    .groupby("month_start_date", as_index=False)
    .agg(
        headcount=("headcount_end", "sum"),
        hires=("hires", "sum"),
        exits=("exits", "sum")
    )
    .sort_values("month_start_date")
)

start_count = int(monthly_trend.iloc[0]["headcount"])
end_count = int(monthly_trend.iloc[-1]["headcount"])
growth_pct = (end_count - start_count) / start_count * 100

# Plot the trend
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    monthly_trend["month_start_date"],
    monthly_trend["headcount"],
    marker="o",
    markersize=3,
    linewidth=2
)

ax.set_title(
    f"Headcount grew {growth_pct:.1f}%: "
    f"{start_count} to {end_count} employees",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Month")
ax.set_ylabel("Month-end headcount")
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(
    f"Insight: Headcount increased by {end_count - start_count} employees "
    f"({growth_pct:.2f}%) from "
    f"{monthly_trend.iloc[0]['month_start_date']:%B %Y} to "
    f"{monthly_trend.iloc[-1]['month_start_date']:%B %Y}."
)

## Monthly hires vs. Exits chart

In [ ]:
import matplotlib.pyplot as plt

# Aggregate monthly hires and exits across departments
hiring_trend = (
    headcount_monthly
    .groupby("month_start_date", as_index=False)
    .agg(
        hires=("hires", "sum"),
        exits=("exits", "sum")
    )
    .sort_values("month_start_date")
)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    hiring_trend["month_start_date"],
    hiring_trend["hires"],
    label="Hires",
    linewidth=2
)

ax.plot(
    hiring_trend["month_start_date"],
    hiring_trend["exits"],
    label="Exits",
    linewidth=2
)

ax.set_title(
    "Monthly hiring and exits show workforce movement",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Month")
ax.set_ylabel("Number of employees")
ax.legend()
ax.grid(True, alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("Total hires:", int(hiring_trend["hires"].sum()))
print("Total exits:", int(hiring_trend["exits"].sum()))
print(
    "Net movement (hires minus exits):",
    int(hiring_trend["hires"].sum() - hiring_trend["exits"].sum())
)

## Department headcount chart


In [ ]:
import matplotlib.pyplot as plt

# Get department names
department_mix = latest_data[
    ["department_id", "headcount_end"]
].merge(
    con.execute("""
        SELECT department_id, department_name
        FROM warehouse.dim_department
    """).fetchdf(),
    on="department_id",
    how="left"
)

department_mix = department_mix.sort_values(
    "headcount_end", ascending=True
)

fig, ax = plt.subplots(figsize=(11, 7))

bars = ax.barh(
    department_mix["department_name"],
    department_mix["headcount_end"]
)

ax.set_title(
    "Engineering leads workforce size with 167 employees",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Headcount at September 2026 month-end")
ax.set_ylabel("Department")
ax.grid(axis="x", alpha=0.3)

# Add values at the end of each bar
ax.bar_label(bars, padding=3)

plt.tight_layout()
plt.show()

largest_department = department_mix.loc[
    department_mix["headcount_end"].idxmax()
]

print(
    f"Insight: {largest_department['department_name']} is the largest "
    f"department with {int(largest_department['headcount_end'])} employees."
)

## Location-tier chart

In [ ]:
import matplotlib.pyplot as plt

# Use only the current employee records
current_employees = employees[
    employees["is_current"] == "Y"
].copy()

# Join employees to the location lookup
location_mix = current_employees.merge(
    locations[["work_location", "city_tier"]],
    on="work_location",
    how="left"
)

# Count employees in each city tier
tier_counts = (
    location_mix.groupby("city_tier", dropna=False)
    .size()
    .reset_index(name="employee_count")
)

tier_counts = tier_counts.sort_values(
    "employee_count", ascending=False
)

print("Employee distribution by location tier:")
display(tier_counts)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    tier_counts["city_tier"].fillna("Unknown"),
    tier_counts["employee_count"]
)

ax.set_title(
    "Current workforce distribution across city tiers",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("City tier")
ax.set_ylabel("Number of employees")
ax.grid(axis="y", alpha=0.3)
ax.bar_label(bars, padding=3)

plt.tight_layout()
plt.show()

## Work-mode chart

In [ ]:
import matplotlib.pyplot as plt

# Count current employees by work mode
workmode_counts = (
    current_employees["work_mode"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("work_mode")
    .reset_index(name="employee_count")
)

workmode_counts["share_pct"] = (
    workmode_counts["employee_count"]
    / workmode_counts["employee_count"].sum()
    * 100
)

print("Current workforce by work mode:")
display(workmode_counts)

# Create chart
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    workmode_counts["work_mode"],
    workmode_counts["employee_count"]
)

ax.set_title(
    "Hybrid is the most common work mode",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Work mode")
ax.set_ylabel("Number of current employees")
ax.grid(axis="y", alpha=0.3)
ax.bar_label(bars, padding=3)

plt.tight_layout()
plt.show()

top_mode = workmode_counts.iloc[0]

print(
    f"Insight: {top_mode['work_mode']} is the most common work mode, "
    f"with {int(top_mode['employee_count'])} employees "
    f"({top_mode['share_pct']:.2f}% of current employee records)."
)

## KPI summary

In [ ]:
import matplotlib.pyplot as plt

# Latest monthly totals
latest_month = headcount_monthly["month_start_date"].max()

latest_data = headcount_monthly[
    headcount_monthly["month_start_date"] == latest_month
]

latest_headcount = int(latest_data["headcount_end"].sum())
latest_hires = int(latest_data["hires"].sum())
latest_exits = int(latest_data["exits"].sum())

first_month = headcount_monthly["month_start_date"].min()
first_data = headcount_monthly[
    headcount_monthly["month_start_date"] == first_month
]

first_headcount = int(first_data["headcount_end"].sum())
net_change = latest_headcount - first_headcount
growth_pct = net_change / first_headcount * 100

# Weighted overall monthly attrition rate
monthly_start = latest_data["headcount_start"].sum()
monthly_exits = latest_data["exits"].sum()

attrition_pct = (
    monthly_exits / monthly_start * 100
    if monthly_start else 0
)

kpis = [
    ("Month-end headcount", f"{latest_headcount:,}"),
    ("Net headcount change", f"+{net_change:,}"),
    ("Headcount growth", f"{growth_pct:.2f}%"),
    ("Monthly hires", f"{latest_hires:,}"),
    ("Monthly exits", f"{latest_exits:,}"),
    ("Monthly exit rate", f"{attrition_pct:.2f}%")
]

fig, axes = plt.subplots(2, 3, figsize=(13, 5))
fig.suptitle(
    f"Workforce KPIs — {latest_month:%B %Y}",
    fontsize=16,
    fontweight="bold"
)

for ax, (label, value) in zip(axes.flat, kpis):
    ax.axis("off")
    ax.text(
        0.5, 0.62, value,
        ha="center", va="center",
        fontsize=22, fontweight="bold"
    )
    ax.text(
        0.5, 0.30, label,
        ha="center", va="center",
        fontsize=10
    )

plt.tight_layout()
plt.show()

print("Reporting month:", latest_month.strftime("%B %Y"))
print("Headcount growth:", f"{growth_pct:.2f}%")

In [ ]:
%pip install ipywidgets

In [ ]:
import ipywidgets as widgets

print("ipywidgets version:", widgets.__version__)
print("Installation successful!")

## Department and Date filters

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Department options
department_lookup = con.execute("""
    SELECT department_id, department_name
    FROM warehouse.dim_department
    ORDER BY department_name
""").fetchdf()

department_options = [
    ("All departments", "ALL")
] + [
    (row["department_name"], row["department_id"])
    for _, row in department_lookup.iterrows()
]

department_filter = widgets.Dropdown(
    options=department_options,
    value="ALL",
    description="Department:",
    style={"description_width": "initial"}
)

# Month options
months = sorted(
    headcount_monthly["month_start_date"].dropna().unique()
)

month_options = [
    (pd.Timestamp(month).strftime("%B %Y"), pd.Timestamp(month))
    for month in months
]

date_filter = widgets.Dropdown(
    options=month_options,
    value=pd.Timestamp(months[-1]),
    description="Reporting month:",
    style={"description_width": "initial"}
)

output = widgets.Output()

def update_dashboard(change=None):
    with output:
        clear_output(wait=True)

        selected_month = pd.Timestamp(date_filter.value)
        selected_department = department_filter.value

        filtered = headcount_monthly[
            headcount_monthly["month_start_date"] == selected_month
        ].copy()

        if selected_department != "ALL":
            filtered = filtered[
                filtered["department_id"] == selected_department
            ]

        print("Reporting month:", selected_month.strftime("%B %Y"))
        print("Department:",
              "All departments" if selected_department == "ALL"
              else department_lookup.loc[
                  department_lookup["department_id"] == selected_department,
                  "department_name"
              ].iloc[0])

        print("Headcount at start:", int(filtered["headcount_start"].sum()))
        print("Hires:", int(filtered["hires"].sum()))
        print("Exits:", int(filtered["exits"].sum()))
        print("Headcount at end:", int(filtered["headcount_end"].sum()))

department_filter.observe(update_dashboard, names="value")
date_filter.observe(update_dashboard, names="value")

display(widgets.HBox([department_filter, date_filter]))
display(output)

update_dashboard()

In [ ]:
# PQ-05 — Dashboard data validation

print("WORKFORCE DASHBOARD VALIDATION")
print("-" * 40)

print("Latest reporting month:",
      latest_month.strftime("%B %Y"))

print("Latest month-end headcount:",
      latest_headcount)

print("Headcount growth:",
      f"{growth_pct:.2f}%")

print("Current employee records:",
      int((employees["is_current"] == "Y").sum()))

print("Monthly trend records:",
      len(monthly_trend))

print("Department chart records:",
      len(department_mix))

print("Location-tier chart records:",
      len(tier_counts))

print("Work-mode chart records:",
      len(workmode_counts))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

# ---------- Prepare dashboard data ----------

monthly = (
    headcount_monthly
    .groupby("month_start_date", as_index=False)
    .agg(
        headcount=("headcount_end", "sum"),
        hires=("hires", "sum"),
        exits=("exits", "sum")
    )
    .sort_values("month_start_date")
)

first_month = monthly.iloc[0]
last_month = monthly.iloc[-1]

start_hc = int(first_month["headcount"])
end_hc = int(last_month["headcount"])
growth = (end_hc - start_hc) / start_hc * 100

latest_date = pd.Timestamp(last_month["month_start_date"])
latest_month_data = headcount_monthly[
    headcount_monthly["month_start_date"] == latest_date
]

latest_hires = int(latest_month_data["hires"].sum())
latest_exits = int(latest_month_data["exits"].sum())
latest_start = int(latest_month_data["headcount_start"].sum())

exit_rate = (
    latest_exits / latest_start * 100
    if latest_start else 0
)

# ---------- Consistent dashboard styling ----------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 9
})

fig = plt.figure(figsize=(17, 16), facecolor="#F3F6FA")
gs = fig.add_gridspec(
    5, 6,
    height_ratios=[0.65, 1.0, 2.2, 2.2, 1.7],
    hspace=0.65,
    wspace=0.65
)

# ---------- Header ----------

header = fig.add_subplot(gs[0, :])
header.axis("off")

header.text(
    0, 0.72,
    "WORKFORCE & HEADCOUNT",
    fontsize=23, fontweight="bold",
    color="#17324D"
)

header.text(
    0, 0.20,
    f"Executive dashboard  |  Reporting month: "
    f"{latest_date:%B %Y}",
    fontsize=11, color="#526579"
)

# ---------- KPI cards ----------

kpis = [
    ("MONTH-END HEADCOUNT", f"{end_hc:,}"),
    ("NET HEADCOUNT CHANGE", f"{end_hc - start_hc:+,}"),
    ("HEADCOUNT GROWTH", f"{growth:.1f}%"),
    ("MONTHLY HIRES", f"{latest_hires:,}"),
    ("MONTHLY EXITS", f"{latest_exits:,}"),
    ("MONTHLY EXIT RATE", f"{exit_rate:.2f}%")
]

for i, (label, value) in enumerate(kpis):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor("white")
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_color("#DCE4ED")

    ax.text(
        0.5, 0.65, value,
        ha="center", va="center",
        fontsize=20, fontweight="bold",
        color="#17324D",
        transform=ax.transAxes
    )

    ax.text(
        0.5, 0.22, label,
        ha="center", va="center",
        fontsize=8, color="#526579",
        wrap=True, transform=ax.transAxes
    )

# ---------- Chart 1: Headcount trend ----------

ax1 = fig.add_subplot(gs[2, :3])
ax1.plot(
    monthly["month_start_date"],
    monthly["headcount"],
    linewidth=2.5
)
ax1.set_title(
    f"Headcount grew {growth:.1f}% "
    f"from {start_hc} to {end_hc}"
)
ax1.set_ylabel("Month-end headcount")
ax1.grid(alpha=0.25)
ax1.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax1.tick_params(axis="x", rotation=25)

ax1.text(
    0.02, -0.27,
    f"Insight: Net increase of {end_hc - start_hc} employees "
    f"from {first_month['month_start_date']:%b %Y} "
    f"to {latest_date:%b %Y}.",
    transform=ax1.transAxes, fontsize=9, wrap=True
)

# ---------- Chart 2: Hires vs exits ----------

ax2 = fig.add_subplot(gs[2, 3:])
ax2.plot(monthly["month_start_date"], monthly["hires"], label="Hires")
ax2.plot(monthly["month_start_date"], monthly["exits"], label="Exits")
ax2.set_title("Monthly hiring and employee exits")
ax2.set_ylabel("Employees")
ax2.legend(frameon=False)
ax2.grid(alpha=0.25)
ax2.tick_params(axis="x", rotation=25)

ax2.text(
    0.02, -0.27,
    f"September: {latest_hires} hires and {latest_exits} exits.",
    transform=ax2.transAxes, fontsize=9
)

# ---------- Chart 3: Department headcount ----------

ax3 = fig.add_subplot(gs[3, :3])

dept_plot = department_mix.sort_values("headcount_end")

ax3.barh(
    dept_plot["department_name"],
    dept_plot["headcount_end"]
)
ax3.set_title("Engineering has the largest workforce")
ax3.set_xlabel("September 2026 month-end headcount")
ax3.grid(axis="x", alpha=0.25)

ax3.text(
    0.02, -0.24,
    "Insight: Engineering accounts for approximately "
    "37.9% of September month-end headcount.",
    transform=ax3.transAxes, fontsize=9, wrap=True
)

# ---------- Chart 4: Location tier ----------

ax4 = fig.add_subplot(gs[3, 3:])

tier_plot = tier_counts.sort_values("city_tier")

ax4.bar(
    tier_plot["city_tier"].fillna("Unknown"),
    tier_plot["employee_count"]
)
ax4.set_title("Current employee records by city tier")
ax4.set_ylabel("Employee records")
ax4.grid(axis="y", alpha=0.25)

ax4.text(
    0.02, -0.24,
    "Population: 745 current employee records, "
    "not the 441 monthly headcount.",
    transform=ax4.transAxes, fontsize=9, wrap=True
)

# ---------- Chart 5: Work mode ----------

ax5 = fig.add_subplot(gs[4, :3])

mode_plot = workmode_counts.sort_values(
    "employee_count", ascending=False
)

ax5.bar(
    mode_plot["work_mode"],
    mode_plot["employee_count"]
)
ax5.set_title("Hybrid is the most common recorded work mode")
ax5.set_ylabel("Employee records")
ax5.grid(axis="y", alpha=0.25)

ax5.text(
    0.02, -0.24,
    "Population: 745 current employee records.",
    transform=ax5.transAxes, fontsize=9
)

# ---------- Executive summary ----------

summary = fig.add_subplot(gs[4, 3:])
summary.set_facecolor("white")
summary.set_xticks([])
summary.set_yticks([])

for spine in summary.spines.values():
    spine.set_color("#DCE4ED")

summary.text(
    0.05, 0.88,
    "EXECUTIVE SUMMARY",
    fontsize=12, fontweight="bold",
    color="#17324D", transform=summary.transAxes
)

summary_lines = [
    f"• Headcount reached {end_hc} in {latest_date:%B %Y}.",
    f"• Long-term headcount growth: {growth:.1f}%.",
    f"• Engineering is the largest department.",
    "• Hybrid is the most common recorded work mode.",
    "• Composition charts use a separate employee population."
]

summary.text(
    0.05, 0.70,
    "\n\n".join(summary_lines),
    fontsize=9, color="#34495E",
    va="top", transform=summary.transAxes,
    wrap=True
)

fig.subplots_adjust(top=0.96, bottom=0.06)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Department and month options
department_lookup = con.execute("""
    SELECT department_id, department_name
    FROM warehouse.dim_department
    ORDER BY department_name
""").fetchdf()

department_filter = widgets.Dropdown(
    options=[("All departments", "ALL")] + [
        (r["department_name"], r["department_id"])
        for _, r in department_lookup.iterrows()
    ],
    value="ALL",
    description="Department",
    layout=widgets.Layout(width="300px")
)

available_months = sorted(
    pd.to_datetime(headcount_monthly["month_start_date"].dropna()).unique()
)

date_filter = widgets.Dropdown(
    options=[
        (pd.Timestamp(m).strftime("%B %Y"), pd.Timestamp(m))
        for m in available_months
    ],
    value=pd.Timestamp(available_months[-1]),
    description="Month",
    layout=widgets.Layout(width="260px")
)

dashboard_output = widgets.Output()

def render_workforce_dashboard(change=None):
    with dashboard_output:
        clear_output(wait=True)

        month = pd.Timestamp(date_filter.value)
        dept = department_filter.value

        # Filter monthly facts by reporting month and department
        monthly_rows = headcount_monthly[
            pd.to_datetime(headcount_monthly["month_start_date"]) == month
        ].copy()

        if dept != "ALL":
            monthly_rows = monthly_rows[
                monthly_rows["department_id"] == dept
            ]

        # Filter the time series by department
        trend_rows = headcount_monthly.copy()
        if dept != "ALL":
            trend_rows = trend_rows[
                trend_rows["department_id"] == dept
            ]

        trend = (
            trend_rows.groupby("month_start_date", as_index=False)
            .agg(
                headcount=("headcount_end", "sum"),
                hires=("hires", "sum"),
                exits=("exits", "sum")
            )
            .sort_values("month_start_date")
        )

        ending = int(monthly_rows["headcount_end"].sum())
        starting = int(monthly_rows["headcount_start"].sum())
        hires = int(monthly_rows["hires"].sum())
        exits = int(monthly_rows["exits"].sum())

        if len(trend) > 1 and trend.iloc[0]["headcount"] != 0:
            first_hc = int(trend.iloc[0]["headcount"])
            growth = (ending - first_hc) / first_hc * 100
        else:
            growth = 0.0

        exit_rate = exits / starting * 100 if starting else 0

        # Department mix for the selected month
        dept_mix = monthly_rows.merge(
            department_lookup,
            on="department_id",
            how="left"
        ).sort_values("headcount_end", ascending=True)

        # Header and KPI cards
        fig = plt.figure(figsize=(15, 11), facecolor="#F3F6FA")
        gs = fig.add_gridspec(
            4, 4,
            height_ratios=[0.5, 0.8, 2.0, 1.7],
            hspace=0.65,
            wspace=0.5
        )

        header = fig.add_subplot(gs[0, :])
        header.axis("off")
        header.text(
            0, 0.75, "WORKFORCE & HEADCOUNT",
            fontsize=21, fontweight="bold", color="#17324D"
        )
        header.text(
            0, 0.18,
            f"Reporting month: {month:%B %Y} | "
            + ("All departments" if dept == "ALL" else
               department_lookup.loc[
                   department_lookup["department_id"] == dept,
                   "department_name"
               ].iloc[0]),
            fontsize=11, color="#526579"
        )

        kpis = [
            ("MONTH-END HEADCOUNT", f"{ending:,}"),
            ("HEADCOUNT CHANGE", f"{ending-starting:+,}"),
            ("MONTHLY HIRES", f"{hires:,}"),
            ("MONTHLY EXITS", f"{exits:,}")
        ]

        for i, (label, value) in enumerate(kpis):
            ax = fig.add_subplot(gs[1, i])
            ax.set_facecolor("white")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_color("#DCE4ED")
            ax.text(
                0.5, 0.63, value,
                ha="center", va="center",
                fontsize=20, fontweight="bold",
                color="#17324D", transform=ax.transAxes
            )
            ax.text(
                0.5, 0.2, label,
                ha="center", va="center",
                fontsize=8, color="#526579",
                transform=ax.transAxes
            )

        # Headcount trend
        ax1 = fig.add_subplot(gs[2, :2])
        ax1.plot(trend["month_start_date"], trend["headcount"], linewidth=2)
        ax1.set_title("Month-end headcount trend", fontweight="bold")
        ax1.set_ylabel("Employees")
        ax1.grid(alpha=0.25)
        ax1.tick_params(axis="x", rotation=25)

        # Hires vs exits
        ax2 = fig.add_subplot(gs[2, 2:])
        ax2.plot(trend["month_start_date"], trend["hires"], label="Hires")
        ax2.plot(trend["month_start_date"], trend["exits"], label="Exits")
        ax2.set_title("Monthly hires and exits", fontweight="bold")
        ax2.set_ylabel("Employees")
        ax2.legend(frameon=False)
        ax2.grid(alpha=0.25)
        ax2.tick_params(axis="x", rotation=25)

        # Department breakdown
        ax3 = fig.add_subplot(gs[3, :2])
        if not dept_mix.empty:
            bars = ax3.barh(
                dept_mix["department_name"],
                dept_mix["headcount_end"]
            )
            ax3.bar_label(bars, padding=3)
        ax3.set_title("Headcount by department", fontweight="bold")
        ax3.set_xlabel("Month-end headcount")
        ax3.grid(axis="x", alpha=0.25)

        # Population note
        ax4 = fig.add_subplot(gs[3, 2:])
        ax4.axis("off")
        ax4.text(
            0, 0.95, "WORKFORCE COMPOSITION",
            fontsize=12, fontweight="bold", color="#17324D"
        )
        ax4.text(
            0, 0.72,
            "Location-tier and work-mode charts use 745 current "
            "employee records from dim_employee.\n\n"
            "They are not recalculated by the monthly department/date "
            "filters and should not be compared directly with monthly "
            "headcount until the populations are reconciled.",
            fontsize=10, color="#34495E",
            va="top", wrap=True
        )

        fig.subplots_adjust(top=0.94, bottom=0.07)
        plt.show()

        print(
            f"Insight: For {month:%B %Y}, the selected population has "
            f"{ending} employees at month-end, {hires} hires and {exits} exits."
        )
        print(f"Monthly exit rate: {exit_rate:.2f}%")

department_filter.observe(render_workforce_dashboard, names="value")
date_filter.observe(render_workforce_dashboard, names="value")

display(
    widgets.VBox([
        widgets.HTML("<h3 style='margin-bottom:4px'>Dashboard filters</h3>"),
        widgets.HBox([department_filter, date_filter]),
        dashboard_output
    ])
)

render_workforce_dashboard()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ---------- Dashboard filters ----------

dept_lookup = con.execute("""
    SELECT department_id, department_name
    FROM warehouse.dim_department
    ORDER BY department_name
""").fetchdf()

dept_filter = widgets.Dropdown(
    options=[("All departments", "ALL")] + [
        (r.department_name, r.department_id)
        for r in dept_lookup.itertuples()
    ],
    value="ALL",
    description="Department:",
    layout=widgets.Layout(width="330px")
)

month_values = sorted(
    pd.to_datetime(headcount_monthly["month_start_date"]).dropna().unique()
)

month_filter = widgets.Dropdown(
    options=[
        (pd.Timestamp(m).strftime("%B %Y"), pd.Timestamp(m))
        for m in month_values
    ],
    value=pd.Timestamp(month_values[-1]),
    description="Month:",
    layout=widgets.Layout(width="290px")
)

dashboard = widgets.Output()

def draw_dashboard(change=None):
    with dashboard:
        dashboard.clear_output(wait=True)

        selected_month = pd.Timestamp(month_filter.value)
        selected_dept = dept_filter.value

        # Filter monthly workforce facts
        facts = headcount_monthly.copy()
        facts["month_start_date"] = pd.to_datetime(facts["month_start_date"])

        if selected_dept != "ALL":
            facts = facts[facts["department_id"] == selected_dept]

        selected = facts[facts["month_start_date"] == selected_month]

        # Monthly time series
        trend = (
            facts.groupby("month_start_date", as_index=False)
            .agg(
                headcount=("headcount_end", "sum"),
                hires=("hires", "sum"),
                exits=("exits", "sum")
            )
            .sort_values("month_start_date")
        )

        start_hc = int(trend.iloc[0]["headcount"])
        end_hc = int(selected["headcount_end"].sum())
        starting = int(selected["headcount_start"].sum())
        hires = int(selected["hires"].sum())
        exits = int(selected["exits"].sum())

        growth = (
            (end_hc - start_hc) / start_hc * 100
            if start_hc else 0
        )
        exit_rate = exits / starting * 100 if starting else 0

        # Department mix for selected month
        dept_data = selected.merge(
            dept_lookup, on="department_id", how="left"
        ).sort_values("headcount_end")

        # ---------- Layout: header, 6 KPIs, 5 charts ----------
        fig = plt.figure(figsize=(18, 20), facecolor="#F3F6FA")
        gs = fig.add_gridspec(
            5, 6,
            height_ratios=[0.5, 0.85, 2.0, 2.0, 1.8],
            hspace=0.85,
            wspace=0.75
        )

        # Header
        header = fig.add_subplot(gs[0, :])
        header.axis("off")
        header.text(
            0, 0.72, "WORKFORCE & HEADCOUNT DASHBOARD",
            fontsize=22, fontweight="bold", color="#17324D"
        )
        dept_label = (
            "All departments" if selected_dept == "ALL"
            else dept_lookup.loc[
                dept_lookup["department_id"] == selected_dept,
                "department_name"
            ].iloc[0]
        )
        header.text(
            0, 0.18,
            f"Reporting month: {selected_month:%B %Y}  |  "
            f"Department: {dept_label}",
            fontsize=11, color="#526579"
        )

        # Six KPI cards
        kpis = [
            ("MONTH-END HEADCOUNT", f"{end_hc:,}"),
            ("NET CHANGE", f"{end_hc - start_hc:+,}"),
            ("HEADCOUNT GROWTH", f"{growth:.1f}%"),
            ("MONTHLY HIRES", f"{hires:,}"),
            ("MONTHLY EXITS", f"{exits:,}"),
            ("MONTHLY EXIT RATE", f"{exit_rate:.2f}%")
        ]

        for i, (label, value) in enumerate(kpis):
            ax = fig.add_subplot(gs[1, i])
            ax.set_facecolor("white")
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_color("#DCE4ED")
            ax.text(
                0.5, 0.62, value, ha="center", va="center",
                fontsize=19, fontweight="bold", color="#17324D",
                transform=ax.transAxes
            )
            ax.text(
                0.5, 0.20, label, ha="center", va="center",
                fontsize=8, color="#526579", wrap=True,
                transform=ax.transAxes
            )

        # Chart 1: Headcount trend
        ax1 = fig.add_subplot(gs[2, :3])
        ax1.plot(trend["month_start_date"], trend["headcount"], linewidth=2)
        ax1.set_title("1. Month-end headcount trend", fontweight="bold")
        ax1.set_ylabel("Employees")
        ax1.grid(alpha=0.25)
        ax1.tick_params(axis="x", rotation=25)

        # Chart 2: Hires vs exits
        ax2 = fig.add_subplot(gs[2, 3:])
        ax2.plot(trend["month_start_date"], trend["hires"], label="Hires")
        ax2.plot(trend["month_start_date"], trend["exits"], label="Exits")
        ax2.set_title("2. Monthly hires vs exits", fontweight="bold")
        ax2.set_ylabel("Employees")
        ax2.legend(frameon=False)
        ax2.grid(alpha=0.25)
        ax2.tick_params(axis="x", rotation=25)

        # Chart 3: Department headcount
        ax3 = fig.add_subplot(gs[3, :3])
        bars = ax3.barh(
            dept_data["department_name"],
            dept_data["headcount_end"]
        )
        ax3.bar_label(bars, padding=2)
        ax3.set_title("3. Headcount by department", fontweight="bold")
        ax3.set_xlabel("Month-end headcount")
        ax3.grid(axis="x", alpha=0.25)

        # Chart 4: Location tiers
        ax4 = fig.add_subplot(gs[3, 3:])
        tier_plot = tier_counts.sort_values("city_tier")
        bars = ax4.bar(
            tier_plot["city_tier"].fillna("Unknown"),
            tier_plot["employee_count"]
        )
        ax4.bar_label(bars, padding=2)
        ax4.set_title("4. Workforce by city tier", fontweight="bold")
        ax4.set_ylabel("Current employee records")
        ax4.grid(axis="y", alpha=0.25)

        # Chart 5: Work modes
        ax5 = fig.add_subplot(gs[4, :3])
        mode_plot = workmode_counts.sort_values(
            "employee_count", ascending=False
        )
        bars = ax5.bar(
            mode_plot["work_mode"],
            mode_plot["employee_count"]
        )
        ax

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["Hires", "Exits"], [17, 3])
ax.set_title("Dashboard display test")
plt.show()

print("Matplotlib display test completed.")
print("ipywidgets version:", widgets.__version__)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

test_slider = widgets.IntSlider(
    value=5,
    min=0,
    max=10,
    description="Widget test:"
)

display(test_slider)

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    ["Headcount", "Hires", "Exits"],
    [441, 17, 3]
)

ax.set_title("Workforce Dashboard — Display Test")
ax.set_ylabel("Employees")

plt.tight_layout()
plt.show()

In [ ]:

%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# --------------------------------------------------
# 1. LOAD LOOKUP DATA
# --------------------------------------------------

dept_lookup = con.execute("""
    SELECT department_id, department_name
    FROM warehouse.dim_department
    ORDER BY department_name
""").fetchdf()

# Current active employee records by job level
job_level_counts = con.execute("""
    SELECT
        job_level,
        COUNT(*) AS employee_count
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    GROUP BY job_level
    ORDER BY job_level
""").fetchdf()

# Ensure date columns use datetime format
headcount_monthly = headcount_monthly.copy()
headcount_monthly["month_start_date"] = pd.to_datetime(
    headcount_monthly["month_start_date"]
)

# --------------------------------------------------
# 2. CREATE DASHBOARD FILTERS
# --------------------------------------------------

dept_filter = widgets.Dropdown(
    options=[("All departments", "ALL")] + [
        (row["department_name"], row["department_id"])
        for _, row in dept_lookup.iterrows()
    ],
    value="ALL",
    description="Department:"
)

month_list = sorted(
    headcount_monthly["month_start_date"].dropna().unique()
)

month_filter = widgets.Dropdown(
    options=[
        (pd.Timestamp(m).strftime("%B %Y"), pd.Timestamp(m))
        for m in month_list
    ],
    value=pd.Timestamp(month_list[-1]),
    description="Month:"
)

# --------------------------------------------------
# 3. DASHBOARD FUNCTION
# --------------------------------------------------

def draw_dashboard(department="ALL", month=None):

    if month is None:
        month = pd.Timestamp(month_list[-1])

    month = pd.Timestamp(month)

    facts = headcount_monthly.copy()

    if department != "ALL":
        facts = facts[
            facts["department_id"] == department
        ]

    month_data = facts[
        facts["month_start_date"] == month
    ]

    if month_data.empty:
        print("No headcount data available for this selection.")
        return

    # Trend data
    trend = (
        facts.groupby("month_start_date", as_index=False)
        .agg(
            headcount=("headcount_end", "sum"),
            hires=("hires", "sum"),
            exits=("exits", "sum")
        )
        .sort_values("month_start_date")
    )

    # KPI calculations
    ending = int(month_data["headcount_end"].sum())
    starting = int(month_data["headcount_start"].sum())
    hires = int(month_data["hires"].sum())
    exits = int(month_data["exits"].sum())
    net_change = ending - starting

    # Growth relative to the first available month
    first_hc = int(trend.iloc[0]["headcount"])
    growth = (
        (ending - first_hc) / first_hc * 100
        if first_hc else 0
    )

    exit_rate = (
        exits / starting * 100
        if starting else 0
    )

    # Department breakdown
    dept_data = (
        month_data.merge(
            dept_lookup,
            on="department_id",
            how="left"
        )
        .sort_values("headcount_end")
    )

    # --------------------------------------------------
    # 4. CREATE FIGURE AND GRID
    # --------------------------------------------------

    fig = plt.figure(
        figsize=(17, 22),
        facecolor="#F3F6FA"
    )

    gs = fig.add_gridspec(
        6, 6,
        height_ratios=[0.55, 0.9, 2, 2.1, 1.8, 1.8],
        hspace=0.9,
        wspace=0.7
    )

    # Dashboard header
    header = fig.add_subplot(gs[0, :])
    header.axis("off")

    header.text(
        0, 0.72,
        "WORKFORCE & HEADCOUNT DASHBOARD",
        fontsize=21,
        fontweight="bold",
        color="#17324D"
    )

    if department == "ALL":
        dept_name = "All departments"
    else:
        matching = dept_lookup.loc[
            dept_lookup["department_id"] == department,
            "department_name"
        ]
        dept_name = (
            matching.iloc[0]
            if not matching.empty else department
        )

    header.text(
        0, 0.15,
        f"{month:%B %Y} | {dept_name}",
        fontsize=11,
        color="#526579"
    )

    # --------------------------------------------------
    # 5. SIX KPI CARDS
    # --------------------------------------------------

    kpis = [
        ("MONTH-END HEADCOUNT", f"{ending:,}"),
        ("NET CHANGE", f"{net_change:+,}"),
        ("HEADCOUNT GROWTH", f"{growth:.1f}%"),
        ("MONTHLY HIRES", f"{hires:,}"),
        ("MONTHLY EXITS", f"{exits:,}"),
        ("MONTHLY EXIT RATE", f"{exit_rate:.2f}%")
    ]

    for i, (label, value) in enumerate(kpis):

        ax = fig.add_subplot(gs[1, i])
        ax.set_facecolor("white")
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_color("#DCE4ED")

        ax.text(
            0.5, 0.63, value,
            ha="center",
            va="center",
            fontsize=18,
            fontweight="bold",
            color="#17324D",
            transform=ax.transAxes
        )

        ax.text(
            0.5, 0.18, label,
            ha="center",
            va="center",
            fontsize=7.5,
            color="#526579",
            wrap=True,
            transform=ax.transAxes
        )

    # --------------------------------------------------
    # 6. CHART 1 — HEADCOUNT TREND
    # --------------------------------------------------

    ax1 = fig.add_subplot(gs[2, :3])

    ax1.plot(
        trend["month_start_date"],
        trend["headcount"],
        linewidth=2
    )

    ax1.set_title(
        "1. Month-end headcount trend",
        fontweight="bold"
    )
    ax1.set_ylabel("Employees")
    ax1.grid(alpha=0.25)
    ax1.tick_params(axis="x", rotation=25)

    # --------------------------------------------------
    # 7. CHART 2 — HIRES VS EXITS
    # --------------------------------------------------

    ax2 = fig.add_subplot(gs[2, 3:])

    ax2.plot(
        trend["month_start_date"],
        trend["hires"],
        label="Hires"
    )

    ax2.plot(
        trend["month_start_date"],
        trend["exits"],
        label="Exits"
    )

    ax2.set_title(
        "2. Monthly hires vs exits",
        fontweight="bold"
    )
    ax2.set_ylabel("Employees")
    ax2.legend(frameon=False)
    ax2.grid(alpha=0.25)
    ax2.tick_params(axis="x", rotation=25)

    # --------------------------------------------------
    # 8. CHART 3 — DEPARTMENT HEADCOUNT
    # --------------------------------------------------

    ax3 = fig.add_subplot(gs[3, :3])

    bars = ax3.barh(
        dept_data["department_name"],
        dept_data["headcount_end"]
    )

    ax3.bar_label(bars, padding=2)

    ax3.set_title(
        "3. Headcount by department",
        fontweight="bold"
    )
    ax3.set_xlabel("Month-end headcount")
    ax3.grid(axis="x", alpha=0.25)

    # --------------------------------------------------
    # 9. CHART 4 — LOCATION TIER
    # --------------------------------------------------

    ax4 = fig.add_subplot(gs[3, 3:])

    tier_plot = tier_counts.sort_values("city_tier")

    bars = ax4.bar(
        tier_plot["city_tier"].fillna("Unknown"),
        tier_plot["employee_count"]
    )

    ax4.bar_label(bars, padding=2)

    ax4.set_title(
        "4. Current employee records by city tier",
        fontweight="bold"
    )
    ax4.set_ylabel("Employee records")
    ax4.grid(axis="y", alpha=0.25)

    # --------------------------------------------------
    # 10. CHART 5 — WORK MODE
    # --------------------------------------------------

    ax5 = fig.add_subplot(gs[4, :3])

    mode_plot = workmode_counts.sort_values(
        "employee_count",
        ascending=False
    )

    bars = ax5.bar(
        mode_plot["work_mode"].fillna("Unknown"),
        mode_plot["employee_count"]
    )

    ax5.bar_label(bars, padding=2)

    ax5.set_title(
        "5. Current employee records by work mode",
        fontweight="bold"
    )
    ax5.set_ylabel("Employee records")
    ax5.grid(axis="y", alpha=0.25)

    # --------------------------------------------------
    # 11. EXECUTIVE INSIGHTS
    # --------------------------------------------------

    ax6 = fig.add_subplot(gs[4, 3:])
    ax6.axis("off")

    ax6.text(
        0, 0.95,
        "EXECUTIVE INSIGHTS",
        fontsize=12,
        fontweight="bold",
        color="#17324D"
    )

    ax6.text(
        0, 0.75,
        f"• Month-end headcount: {ending:,}\n\n"
        f"• Monthly hires: {hires:,}; exits: {exits:,}\n\n"
        f"• Net workforce change: {net_change:+,}\n\n"
        "• Location and work-mode charts use current "
        "employee records.\n\n"
        "• Composition charts use a different population "
        "from the historical monthly headcount.",
        va="top",
        fontsize=9,
        wrap=True
    )

    # --------------------------------------------------
    # 12. CHART 6 — JOB-LEVEL DISTRIBUTION
    # --------------------------------------------------

    ax7 = fig.add_subplot(gs[5, :])

    job_plot = job_level_counts.sort_values("job_level")

    bars = ax7.bar(
        job_plot["job_level"].astype(str),
        job_plot["employee_count"]
    )

    ax7.bar_label(bars, padding=3)

    ax7.set_title(
        "6. Current active employee records by job level",
        fontweight="bold"
    )
    ax7.set_xlabel("Job level")
    ax7.set_ylabel("Employee records")
    ax7.grid(axis="y", alpha=0.25)

    # Layout and display
    fig.subplots_adjust(
        top=0.95,
        bottom=0.04
    )

    plt.show()
    plt.close(fig)


# --------------------------------------------------
# 13. CONNECT FILTERS TO DASHBOARD
# --------------------------------------------------

dashboard_output = widgets.interactive_output(
    draw_dashboard,
    {
        "department": dept_filter,
        "month": month_filter
    }
)

display(
    widgets.VBox([
        widgets.HTML("<h3>Dashboard filters</h3>"),
        widgets.HBox([dept_filter, month_filter]),
        dashboard_output
    ])
)


In [ ]:
from pathlib import Path

project_folder = Path(r"C:\Users\Hp\Documents\pulseiq")

for path in project_folder.iterdir():
    print(path.name)

In [ ]:
con.execute("""
    DESCRIBE warehouse.dim_employee
""").df()

In [ ]:
con.execute("""
    SELECT
        job_level,
        COUNT(*) AS employee_records
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    GROUP BY job_level
    ORDER BY job_level
""").df()

In [ ]:
job_level_counts = con.execute("""
    SELECT
        job_level,
        COUNT(*) AS employee_records
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    GROUP BY job_level
    ORDER BY job_level
""").df()

plt.figure(figsize=(8, 4))

plt.bar(
    job_level_counts["job_level"].astype(str),
    job_level_counts["employee_records"]
)

plt.title("6. Current Active Employee Records by Job Level")
plt.xlabel("Job Level")
plt.ylabel("Employee Records")

for i, value in enumerate(job_level_counts["employee_records"]):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    job_level_counts["job_level"].astype(int).astype(str),
    job_level_counts["employee_records"]
)

plt.title("6. Current Active Employee Records by Job Level")
plt.xlabel("Job Level")
plt.ylabel("Employee Records")

for i, value in enumerate(job_level_counts["employee_records"]):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [ ]:
print(job_level_counts[["job_level", "employee_records"]].to_string(index=False))

print("\nData type:", job_level_counts["job_level"].dtype)

In [ ]:
con.execute("""
    SELECT DISTINCT job_level
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    ORDER BY job_level
""").df()

In [ ]:
job_level_counts = con.execute("""
    SELECT
        CAST(job_level AS INTEGER) AS job_level,
        COUNT(*) AS employee_records
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    GROUP BY job_level
    ORDER BY job_level
""").df()

print(job_level_counts.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    job_level_counts["job_level"].astype(str),
    job_level_counts["employee_records"]
)

plt.title("6. Current Active Employee Records by Job Level")
plt.xlabel("Job Level")
plt.ylabel("Employee Records")

for i, value in enumerate(job_level_counts["employee_records"]):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [ ]:
print("Raw job levels:")
print(
    con.execute("""
        SELECT job_level, COUNT(*) AS total
        FROM warehouse.dim_employee
        WHERE is_current = 'Y'
          AND employment_status = 'Active'
        GROUP BY job_level
        ORDER BY job_level
    """).df().to_string(index=False)
)

print("\nCurrent DataFrame:")
print(job_level_counts.to_string(index=False))

In [ ]:
con.execute("""
    SELECT
        employee_id,
        job_level,
        employment_status,
        is_current
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    ORDER BY job_level
    LIMIT 15
""").df()

In [ ]:
print(con.execute("""
    SELECT
        employee_id,
        job_level,
        employment_status,
        is_current
    FROM warehouse.dim_employee
    WHERE is_current = 'Y'
      AND employment_status = 'Active'
    ORDER BY job_level
    LIMIT 10
""").df().to_string(index=False))